In [1]:
import os
import pandas as pd
import ast
from Bio import SeqIO
import warnings

warnings.filterwarnings('ignore')

def extract_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
def blast_result(genus_name, acc_n, contig, que):
    folder = f'/active-data/analysis_results/chr_pla/genus/cor-pla_fraction_records/{genus_name}/{acc_n}'
    handle = open(f'/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/data/{acc_n}/genomic.gbff')
    acc_record = SeqIO.parse(handle, 'genbank')
    for seq_record in acc_record:
        if seq_record.id == contig:
            os.chdir('/active-data/temp/blastn')
            query_file = open(f'temp_query-{contig}.fasta', 'w+')
            SeqIO.write(seq_record, query_file, "fasta")
            query_file.close()
            len_query = len(seq_record)
            os.system(f'blastn -query temp_query-{contig}.fasta -db /active-data/analysis_results/chr_pla/genus/blast_data/{genus_name}/nucleotide_seq.blastdb -out blastn_results-{contig}.txt -evalue 1e-50 -max_target_seqs 100000 -max_hsps 3000 -outfmt 6 -num_threads 8')
            head = ['qseqid', 'sseqid', 'pident', 'length', 'mismatch', 'gapopen', 'qstart', 'qend', 'sstart', 'send', 'evalue', 'bitscore']
            align_result = pd.read_csv(f'blastn_results-{contig}.txt', sep = '\t', engine = 'python', header = None, names = head)
            os.system(f'rm temp_query-{contig}.fasta')
            os.system(f'rm blastn_results-{contig}.txt')
            os.chdir(folder)
            align_result.to_csv(f'{seq_record.id}_blast_result.csv', index=False)
    handle.close()
    que.put(1)

In [3]:
from tqdm import tqdm
import multiprocessing

label = "pident_90"
for genus_name in keep_genus:
    base_folder = f'/active-data/analysis_results/chr_pla/genus'
    folder = f'{base_folder}/statistics_records/{genus_name}'
    replicon_data = pd.read_csv(f'{folder}/replicon-plasmid_fraction-self_bitscore_statistics.csv')
    IRs = replicon_data[replicon_data[f'category-{label}']=='intermediate replicon'].copy().reset_index(drop=True)
    
    manager = multiprocessing.Manager()
    que = manager.Queue()
    
    par = 10
    tot = len(IRs)
    pool = multiprocessing.Pool(par)

    NMS_count = 0
    for i in IRs.index:
        acc_n, contig = IRs['accession'][i].split('-')
        pool.apply_async(blast_result, (genus_name, acc_n, contig, que))
    
    pool.close()
    
    count = 0
    with tqdm(total = tot, desc=f'{genus_name}', leave=True, ncols=100, unit='B', unit_scale=True) as pbar:
        while True and tot > 0:
            if not que.empty():
                value = que.get(True)
                count += 1
                pbar.update(1)
                if count == tot:
                    break
            else:
                continue
    
    pool.join()

Klebsiella: 100%|███████████████████████████████████████████████████| 150/150 [05:37<00:00, 2.25s/B]
Staphylococcus: 100%|█████████████████████████████████████████████| 78.0/78.0 [00:59<00:00, 1.30B/s]
Pseudomonas: 100%|████████████████████████████████████████████████| 63.0/63.0 [01:58<00:00, 1.88s/B]
Salmonella: 100%|███████████████████████████████████████████████████| 109/109 [01:13<00:00, 1.49B/s]
Streptococcus: 100%|██████████████████████████████████████████████| 19.0/19.0 [00:13<00:00, 1.39B/s]
Streptomyces: 100%|███████████████████████████████████████████████| 64.0/64.0 [02:53<00:00, 2.72s/B]
Acinetobacter: 100%|██████████████████████████████████████████████| 69.0/69.0 [00:31<00:00, 2.17B/s]
Enterococcus: 100%|███████████████████████████████████████████████| 47.0/47.0 [00:23<00:00, 1.97B/s]
Bordetella: 100%|█████████████████████████████████████████████████| 2.00/2.00 [00:02<00:00, 1.37s/B]
Enterobacter: 100%|███████████████████████████████████████████████| 30.0/30.0 [00:17<00:00,